## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from statistics import mean
import warnings
import math

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from scipy.signal import get_window
from scipy.fft import rfft, rfftfreq
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import paths
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = [
    "TB-twentyfive_again_threshold",
    "TB-hundreed_again_threshold",
    "TB-twohundred_fifty_again_threshold",
]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

## Data Loading and Initial Processing

In [ ]:
# Data Loading

for exp_id, exp_data in experiment_neutron_data.items():
    exp_base_path = paths.get_exp_root(exp_id)
    spectra_folder = exp_base_path / "processed_data/unfiltered/spectra"
    time_spectrum_path = spectra_folder / "time_spectrum_0.parquet"
    df = pd.read_parquet(time_spectrum_path)
    print(df.head(200))
    exp_data["time_spectrum"] = df

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
zorders = {
    "TB-twentyfive_again_threshold": 0,
    "TB-hundreed_again_threshold": 0,
    "TB-twohundred_fifty_again_threshold": 0,
}
colors = {
    "TB-twentyfive_again_threshold": bg_red,
    "TB-hundreed_again_threshold": bg_bluegrey,
    "TB-twohundred_fifty_again_threshold": bg_blue,
}

alpha = 1
merge_count = 20
for exp_id, exp_data in experiment_neutron_data.items():
    zorder_mod = zorders[exp_id]
    color = colors[exp_id]

    spectrum_df = exp_data["time_spectrum"]
    bin_starts = spectrum_df["bin_time"].to_numpy()
    bin_counts = spectrum_df["count"].to_numpy()
    if merge_count is not None or merge_count > 1:
        # select every merge_count element of bin_starts
        indices = np.arange(0, bin_counts.size, merge_count)
        bin_starts = bin_starts[indices]
        # sum every merge_count slice of bin_counts
        bin_counts = np.add.reduceat(bin_counts, indices)
        pass
    bin_widths = bin_starts[1:] - bin_starts[:-1]
    suffix = [bin_widths[-1]]
    bin_widths = np.concatenate((bin_widths, suffix))
    
    ax.bar(
        bin_starts,
        bin_counts,
        width=bin_widths,
        align="edge",
        zorder=5+zorder_mod,
        color=color,
        alpha=alpha,
        lw=0
    )
    # ax.fill_between(
    #     spectrum_df["bin_time"],
    #     spectrum_df["count"],
    #     label=exp_id,
    #     # alpha=alpha,
    #     color=color,
    #     zorder=5+zorder_mod,
    # )
ax.set_xlim(0, 500000)
ax.set_ylim(0, 25 * merge_count)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel(r"Time interval ($\mu$s)", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.0f}")
# ax.set_yscale("log")
# ax.legend()